# Spatial Data Cleaning: SSURGO Map-Unit Crosswalk

Cleans the raw SSURGO soil map-unit polygon layer and spatially joins it
against the EPA water-quality monitoring stations to produce a
**station → map unit (`mukey`)** crosswalk keyed on
`MonitoringLocationIdentifier`. This is the spatial glue needed to attach
`ssurgo-iowa-attributes-clean.csv` (soil drainage class, hydrologic group,
Ksat, AWC) to individual stations in a future soil merge step.

**Input:**
- `data/spatial/01_raw/ssurgo/iowa-mapunit-polygons.shp` (2,712,435 map-unit
  polygons across all 99 Iowa counties, WGS84 / EPSG:4326)
- `data/tabular/02_clean/water-quality/epa-stations-clean.csv` (1,666 station points)

**Output:** `data/spatial/02_clean/ssurgo/ssurgo-mapunit-station-crosswalk-clean.csv`

> **Note:** this polygon layer previously only covered 5 of 99 counties on
> disk — the download notebook (`src/01_download/ssurgo-soil-download.ipynb`)
> hardcoded a single WSS export date, which only matched the counties
> certified that day and returned HTTP 400 for the rest. That notebook now
> fetches each county's actual certification date from SDA's `sacatalog`
> table and was re-run to produce the full statewide extract used here.

> **Expected coverage gap:** because water-quality stations sit in or next to
> streams/lakes, roughly a third of them land in a non-soil SSURGO map unit
> (`musym` `W` / `RIVER` / `LAKE`) that has no component/horizon data and so
> never appears in `ssurgo-iowa-attributes-clean.csv`. That's a real property
> of where these stations are, not a join bug — see the Step 4 QA breakdown.

**Cleaning steps:**

1. Load the raw polygons (only the `MUKEY`/`MUSYM`/`AREASYMBOL` columns —
   the geometry is the expensive part, so we skip unused attribute columns
   like `SPATIALVER`/`areasymb_1`). Confirm CRS and repair invalid
   geometries with `shapely.make_valid`.
2. Load the cleaned station table and spatially join each station point to
   the map-unit polygon it falls **within**.
3. **Fallback for unmatched stations** — a handful of stations (typically
   in-stream/lake sampling points sitting just outside the mapped soil
   extent) won't fall within any polygon. For those, snap to the *nearest*
   polygon in a projected CRS, capped at a sanity-check distance, and flag
   the match method.
4. **QA** — assert every station has exactly one final match (no fan-out,
   no remaining unmatched stations within the distance cap), and report
   what fraction of resulting `mukey`s are actually present in
   `ssurgo-iowa-attributes-clean.csv` (and why the rest aren't).
5. Select/order columns, sort by `MonitoringLocationIdentifier`, and write
   the tidy crosswalk to `02_clean`.

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
from pathlib import Path

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "spatial" / "01_raw" / "ssurgo"
STATIONS_PATH = REPO_ROOT / "data" / "tabular" / "02_clean" / "water-quality" / "epa-stations-clean.csv"
ATTRS_PATH = REPO_ROOT / "data" / "tabular" / "02_clean" / "soil" / "ssurgo-iowa-attributes-clean.csv"
CLEAN_DIR = REPO_ROOT / "data" / "spatial" / "02_clean" / "ssurgo"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:  ", REPO_ROOT)
print("Raw dir:    ", RAW_DIR)
print("Stations:   ", STATIONS_PATH)
print("Clean dir:  ", CLEAN_DIR)

Repo root:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:     /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/spatial/01_raw/ssurgo
Stations:    /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/water-quality/epa-stations-clean.csv
Clean dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/spatial/02_clean/ssurgo


## Step 1 — Load & repair the map-unit polygons

`MUKEY` is *not* a unique polygon id — the same map unit type recurs as many
disjoint polygons across the state (11,208 distinct map units spread over
~2.7M polygons) — so unlike the HUC-12 layer there is no polygon-key
uniqueness check here; the join simply finds whichever polygon a station
point physically falls inside.

In [3]:
mapunits = gpd.read_file(
    RAW_DIR / "iowa-mapunit-polygons.shp",
    columns=["MUKEY", "MUSYM", "AREASYMBOL"],
    engine="pyogrio",
)
print(f"Loaded {len(mapunits):,} polygons, CRS = {mapunits.crs}")
print(f"Distinct MUKEY values: {mapunits['MUKEY'].nunique():,}")

n_invalid = (~mapunits.geometry.is_valid).sum()
print(f"Invalid geometries: {n_invalid:,}")
mapunits["geometry"] = mapunits.geometry.apply(
    lambda g: g if g.is_valid else shapely.make_valid(g)
)
assert (~mapunits.geometry.is_valid).sum() == 0, "invalid geometry remains after repair"

Loaded 2,712,435 polygons, CRS = EPSG:4326
Distinct MUKEY values: 11,208


Invalid geometries: 282


## Step 2 — Load stations & spatial join

Build a point GeoDataFrame from the cleaned station table (already WGS84,
matching the polygon CRS — no reprojection needed for this join) and join
each point to the map-unit polygon it falls **within**.

In [4]:
stations = pd.read_csv(STATIONS_PATH)
print(f"Loaded {len(stations):,} stations")

points = gpd.GeoDataFrame(
    stations[["MonitoringLocationIdentifier"]],
    geometry=gpd.points_from_xy(stations["LongitudeMeasure"], stations["LatitudeMeasure"]),
    crs=4326,
)

within_join = gpd.sjoin(
    points,
    mapunits,
    how="left",
    predicate="within",
).drop(columns="index_right")

n_matched = within_join["MUKEY"].notna().sum()
n_unmatched = within_join["MUKEY"].isna().sum()
print(f"Matched within a polygon: {n_matched:,} / {len(stations):,}")
print(f"Unmatched: {n_unmatched:,}")

dup_matches = within_join.groupby("MonitoringLocationIdentifier").size()
assert (dup_matches > 1).sum() == 0, "station matched more than one polygon"

Loaded 1,666 stations


Matched within a polygon: 1,661 / 1,666
Unmatched: 5


## Step 3 — Nearest-polygon fallback for unmatched stations

Stations that don't fall inside any mapped polygon are almost always
in-stream or lake sampling points sitting just off the soil-survey extent
(open water is generally not mapped as a soil map unit). For those, find
the nearest polygon in a projected CRS (EPSG:5070, CONUS Albers — meters)
and accept the match only if it's within `MAX_SNAP_DISTANCE_M` of the
station; beyond that, something would be structurally wrong and we want to
know about it rather than silently snapping to a far-away polygon.

In [5]:
MAX_SNAP_DISTANCE_M = 500

unmatched_ids = within_join.loc[within_join["MUKEY"].isna(), "MonitoringLocationIdentifier"]

if len(unmatched_ids):
    points_m = points.set_index("MonitoringLocationIdentifier").to_crs(5070)
    mapunits_m = mapunits.to_crs(5070)
    unmatched_points_m = points_m.loc[unmatched_ids].reset_index()

    nearest = gpd.sjoin_nearest(
        unmatched_points_m,
        mapunits_m,
        how="left",
        distance_col="dist_m",
    ).drop(columns="index_right")

    print(nearest[["MonitoringLocationIdentifier", "MUKEY", "dist_m"]].to_string(index=False))
    assert (nearest["dist_m"] <= MAX_SNAP_DISTANCE_M).all(), (
        "a station is farther than MAX_SNAP_DISTANCE_M from any mapped polygon"
    )
else:
    nearest = within_join.iloc[0:0].assign(dist_m=pd.Series(dtype=float))
    print("No unmatched stations — nothing to snap.")

                         MonitoringLocationIdentifier   MUKEY     dist_m
                                        USGS-05474500 1681099 112.126069
                                        USGS-06485950  401987   9.245318
EPA_R7_WQX-DeSoto Lake - (DeSoto NWR) - E of Blair,NE 1604083  76.424101
                              EPA_R7_WQX-ND-23-ND-23a  411667   1.632952
                                    SDDENR_WQX-460832 1865941   1.123224


## Step 4 — Combine matches & QA

Union the within-matches (`dist_m = 0`, exact containment) with the
nearest-neighbor fallback matches, tag each row with how it was matched,
and confirm every station now has exactly one row. As an independent
check, report what fraction of the resulting `MUKEY`s are actually present
in `ssurgo-iowa-attributes-clean.csv` — the attribute extract filters to
major soil components, so a handful of spatial-only `MUKEY`s (open water,
pits, borrow areas) are expected not to join.

In [6]:
within_matched = within_join[within_join["MUKEY"].notna()].copy()
within_matched["match_method"] = "within"
within_matched["dist_m"] = 0.0

nearest_matched = nearest.copy()
nearest_matched["match_method"] = "nearest"

crosswalk = pd.concat(
    [
        within_matched[["MonitoringLocationIdentifier", "MUKEY", "MUSYM", "AREASYMBOL", "match_method", "dist_m"]],
        nearest_matched[["MonitoringLocationIdentifier", "MUKEY", "MUSYM", "AREASYMBOL", "match_method", "dist_m"]],
    ],
    ignore_index=True,
)

dup_final = crosswalk["MonitoringLocationIdentifier"].duplicated().sum()
assert dup_final == 0, "duplicate station in final crosswalk"
assert len(crosswalk) == len(stations), "crosswalk row count does not match station count"
print(crosswalk["match_method"].value_counts().to_string())

attrs = pd.read_csv(ATTRS_PATH, dtype={"mukey": str})
attr_mukeys = set(attrs["mukey"])
has_attrs = crosswalk["MUKEY"].isin(attr_mukeys)
print(f"\nCrosswalked MUKEYs present in ssurgo-iowa-attributes-clean.csv: {has_attrs.sum():,} / {len(crosswalk):,}")

# Water-quality stations sit in or next to streams/lakes, which SSURGO maps as
# non-soil map units (musym "W" / "RIVER" / "LAKE") that have no component/
# horizon data and so never appear in the attribute extract. Confirm that's
# actually what's driving the gap above, rather than a join-key mismatch.
print("\nTop map_unit_symbol values among stations with NO attribute match:")
print(crosswalk.loc[~has_attrs, "MUSYM"].value_counts().head(10).to_string())

match_method
within     1661
nearest       5

Crosswalked MUKEYs present in ssurgo-iowa-attributes-clean.csv: 1,077 / 1,666

Top map_unit_symbol values among stations with NO attribute match:
MUSYM
W        459
RIVER     84
LAKE      21
5040      11
504        3
354        2
5070       2
4000       2
4002       1
5010       1


## Step 5 — Select, rename, sort & write

Rename to the snake_case convention, reorder columns to lead with the
station key, sort, and write the tidy crosswalk to `02_clean`.

In [7]:
COLUMN_MAP = {
    "MUKEY": "mukey",
    "MUSYM": "map_unit_symbol",
    "AREASYMBOL": "survey_area",
    "match_method": "match_method",
    "dist_m": "match_distance_m",
}
crosswalk = (
    crosswalk.rename(columns=COLUMN_MAP)
    .loc[:, ["MonitoringLocationIdentifier", *COLUMN_MAP.values()]]
    .sort_values("MonitoringLocationIdentifier")
    .reset_index(drop=True)
)

out_path = CLEAN_DIR / "ssurgo-mapunit-station-crosswalk-clean.csv"
crosswalk.to_csv(out_path, index=False)
print(f"Wrote {len(crosswalk):,} rows × {crosswalk.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
crosswalk.head()

Wrote 1,666 rows × 6 cols to:
  data/spatial/02_clean/ssurgo/ssurgo-mapunit-station-crosswalk-clean.csv


,MonitoringLocationIdentifier,mukey,map_unit_symbol,survey_area,match_method,match_distance_m
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,402414,2670,IA005,within,0.0
1,11NPSWRD_WQX-HTLN_HEHO_HOOV1,1397070,133+,IA031,within,0.0
2,21IOWA_WQX-10030001,402634,W,IA005,within,0.0
3,21IOWA_WQX-10030002,402414,2670,IA005,within,0.0
4,21IOWA_WQX-10040002,1903207,W,IA007,within,0.0
